# Menia — Atelier en contexte pour Qwen3-4B
Protocole : `docs/LLM_ATELIER_PROTOCOL.md` à la révision `bd82c06`.
Runtime A100. Aucun poids n'est entraîné. Conserver `llm-atelier.zip` à la fin.
Ce notebook teste une disposition d'un modèle préentraîné ; il ne teste ni conscience ni entraînement sans information d'origine.


In [ ]:
import subprocess, sys, os, json, hashlib, datetime
COMMIT = 'bd82c06'
if not os.path.exists('/content/Menia'):
    subprocess.run(['git', 'clone', 'https://github.com/speed25200-cyber/Menia.git', '/content/Menia'], check=True)
os.chdir('/content/Menia')
subprocess.run(['git', 'checkout', '--quiet', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==2.2.6', 'transformers==4.56.2', 'accelerate==1.10.1'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'tests_research.test_llm_atelier', '-v'], check=True)
import torch; assert torch.cuda.is_available(), 'Activer le GPU'
from research.llm_atelier import HFResponder, run_plan
from huggingface_hub import HfApi
revision = HfApi().model_info('Qwen/Qwen3-4B').sha
OUT = '/content/llm-atelier'
responder = HFResponder('Qwen/Qwen3-4B', revision)
started = datetime.datetime.now(datetime.timezone.utc).isoformat()
summary = run_plan(responder, OUT, episodes_per_condition=48)
receipt = {'commit': COMMIT, 'model': 'Qwen/Qwen3-4B', 'revision': revision, 'gpu': torch.cuda.get_device_name(0),
           'started': started, 'finished': datetime.datetime.now(datetime.timezone.utc).isoformat(),
           'episodes_sha256': hashlib.sha256(open(OUT + '/episodes.jsonl', 'rb').read()).hexdigest()}
json.dump(receipt, open(OUT + '/receipt.json', 'w'), indent=2)
print(json.dumps(summary, indent=2, ensure_ascii=False)); print(receipt)
subprocess.run(['zip', '-qr', '/content/llm-atelier.zip', OUT], check=True)
print('Télécharger /content/llm-atelier.zip')
